# 2026 COMP90042 Project
*Make sure you change the file name with your group id.*

# Readme
*If there is something to be noted for the marker, please mention here.*

*If you are planning to implement a program with Object Oriented Programming style, please put those the bottom of this ipynb file*

# 1.DataSet Processing
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [1]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DATA_DIR = Path('/content/drive/MyDrive/nlp_project/data')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import json

# read data
with open(DATA_DIR / 'train-claims.json') as f:
    train_claims = json.load(f)

with open(DATA_DIR / 'dev-claims.json') as f:
    dev_claims = json.load(f)

with open(DATA_DIR / 'test-claims-unlabelled.json') as f:
    test_claims = json.load(f)

with open(DATA_DIR / 'evidence.json') as f:
    evidence = json.load(f)

In [3]:
print(f"train: {len(train_claims)}")
print(f"dev: {len(dev_claims)}")
print(f"test: {len(test_claims)}")
print(f"evidence: {len(evidence)}")

train: 1228
dev: 154
test: 153
evidence: 1208827


In [4]:
from collections import Counter
import numpy as np

# 统计训练集标签分布
label_dist = Counter(c['claim_label'] for c in train_claims.values())
print('训练集标签分布:')
for lbl, cnt in label_dist.most_common():
    print(f'    {lbl:20s} {cnt:>5d} ({cnt/len(train_claims):.1%})')

# 统计每条 claim 对应的 Ground Truth (GT) evidence 数量
gt_counts = [len(c['evidences']) for c in train_claims.values()]
print(f'\n每条 claim 的 GT evidence 数:')
print(f'    min={min(gt_counts)}, max={max(gt_counts)},')
print(f'    mean={np.mean(gt_counts):.2f}, median={int(np.median(gt_counts))}')

训练集标签分布:
    SUPPORTS               519 (42.3%)
    NOT_ENOUGH_INFO        386 (31.4%)
    REFUTES                199 (16.2%)
    DISPUTED               124 (10.1%)

每条 claim 的 GT evidence 数:
    min=1, max=5,
    mean=3.36, median=3


# Baseline TFIDF

In [5]:
# # Baseline TFIDF
# import nltk
# nltk.download('stopwords')
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.metrics.pairwise import cosine_similarity
# import numpy as np
# from nltk.corpus import stopwords

# # 尝试了lemma和其他预处理，但是不如只lower case和stopword好
# # 原因：有很多温度，度数符号会被去掉，但是这个很重要，所以效果不好
# def tfidf(evidences, claims):
#     # 初始化向量化器，并设置移除英文停用词和自动转小写
#     vectorizer = TfidfVectorizer(stop_words=stopwords.words('english')) # lowercase remove stop_word

#     # 1. 学习证据库的词汇表并将其转化为 TF-IDF 矩阵
#     evidence_tfidf = vectorizer.fit_transform(evidences)

#     # 2. 使用相同的词汇表将 claims 转化为 TF-IDF 向量
#     claims_tfidf = vectorizer.transform(claims)

#     # 3. 计算 claims 向量与每一个 evidence 向量之间的余弦相似度
#     cos_similarity = cosine_similarity(claims_tfidf, evidence_tfidf)

#     return cos_similarity

In [6]:
# def choose_top_k(cos_similarity, evidence_ids, claim_ids, top_k):
#     # 注意：图片中初始化为列表 [] 但后续用了键值对赋值，建议改为字典 {}
#     predict_tfidf = {}

#     for i, cid in enumerate(claim_ids):
#         cos_row = cos_similarity[i]

#         # np.argpartition 可以高效地找到前 top_k 大的索引（时间复杂度 O(n)）
#         # -cos_row 是为了降序排列
#         top_idx = np.argpartition(-cos_row, top_k)[:top_k]

#         # 将索引转化为实际的 evidence_id
#         predict_tfidf[cid] = [evidence_ids[idx] for idx in top_idx]

#     return predict_tfidf

In [7]:
# 自定义评估方法
def eval_retrieval(claims_dataset, predict):
    all_recalls, all_precisions, all_fscores = [], [], []

    for claim_id, claim in sorted(claims_dataset.items()):
        if claim_id not in predict:
            continue

        evidence_correct = 0
        evidence_recall = 0.0
        evidence_precision = 0.0
        evidence_fscore = 0.0

        # 确保预测结果存在且不为空
        if isinstance(predict[claim_id], list) and len(predict[claim_id]) > 0:
            predict_set = set(predict[claim_id])

            # 计算预测对的证据数量（交集）
            for true_eid in claim['evidences']:
                if true_eid in predict_set:
                    evidence_correct += 1

            if evidence_correct > 0:
                # Recall: 找回了多少比例的正确证据
                evidence_recall = evidence_correct / len(claim['evidences'])
                # Precision: 预测出的证据中有多少是真正正确的
                evidence_precision = evidence_correct / len(predict[claim_id])
                # F1: 精确率和召回率的调和平均
                evidence_fscore = (2 * evidence_precision * evidence_recall) / (evidence_precision + evidence_recall)

        all_recalls.append(evidence_recall)
        all_precisions.append(evidence_precision)
        all_fscores.append(evidence_fscore)

    print(f'Mean Recall:    {np.mean(all_recalls):.6f}')
    print(f'Mean Precision: {np.mean(all_precisions):.6f}')
    print(f'Mean F1-Score:  {np.mean(all_fscores):.6f}')

    return np.mean(all_fscores)

In [8]:
# # 将证据库的 ID 和内容分别提取为列表
# evidence_list_ids    = list(evidence.keys())
# evidence_list_texts  = list(evidence.values())

# # 将开发集（Dev Set）的 ID 提取为列表
# dev_list_ids         = list(dev_claims.keys())

# # 使用列表推导式提取开发集中每一条 claim 的具体文本内容
# dev_list_texts       = [dev_claims[c]['claim_text'] for c in dev_list_ids]

# # 打印检查第二条数据（索引为 1）以确保提取正确
# print(evidence_list_ids[1])
# print(evidence_list_texts[1])
# print(dev_list_ids[1])
# print(dev_list_texts[1])

In [9]:
# # 设置超参数 K，即每条 claim 检索多少条证据
# TOP_K = 5

# print("TFIDF baseline")

# # 1. 计算余弦相似度矩阵
# cos_word = tfidf(evidence_list_texts, dev_list_texts)

# # 2. 根据相似度分值，为每条 claim 筛选出前 K 条证据 ID
# predict_word = choose_top_k(cos_word, evidence_list_ids, dev_list_ids, TOP_K)

# # 3. 评估检索性能（计算 Recall, Precision, F1）
# f_word = eval_retrieval(dev_claims, predict_word)

# Base1_predict = predict_word
# Base1_f = f_word

In [10]:
# 安装 bm25s 库和句子转换器
!pip install -q bm25s sentence-transformers

In [11]:
import time
import bm25s
import psutil

# 提取证据库的 ID 和文本
evidence_ids    = list(evidence.keys())
evidence_texts  = [evidence[eid] for eid in evidence_ids]

print('Tokenizing 1.2M evidence ...')
t0 = time.time()

# 对证据进行分词处理（Tokenization）
# 使用英文停用词过滤，暂不使用词干提取（stemmer=None）
corpus_tokens = bm25s.tokenize(evidence_texts, stopwords='en', stemmer=None)

print(f'  tokenize 用时: {time.time() - t0:.0f}s')

print('\nBuilding BM25 index ...')
t0 = time.time()

# 初始化 BM25 检索器并构建索引
retriever = bm25s.BM25()
retriever.index(corpus_tokens)

print(f'  index 用时: {time.time() - t0:.0f}s')

# base1用的gpu，bm25用的是cpu

Tokenizing 1.2M evidence ...


Split strings:   0%|          | 0/1208827 [00:00<?, ?it/s]

DEBUG:bm25s:Building index from IDs objects


  tokenize 用时: 18s

Building BM25 index ...


BM25S Count Tokens:   0%|          | 0/1208827 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/1208827 [00:00<?, ?it/s]

  index 用时: 32s


In [12]:
from tqdm.auto import tqdm

def bm25s_retrieve(claims_dict, k=100):
    # 提取所有 claim 的 ID 和文本内容
    cids  = list(claims_dict.keys())
    texts = [claims_dict[c]['claim_text'] for c in cids]

    # 对查询（claims）进行分词处理
    query_tokens = bm25s.tokenize(texts, stopwords='en', stemmer=None)
    # 和之前的分词要相同

    # 在之前构建好的索引（retriever）中进行检索
    # results 会返回匹配到的证据索引，scores 返回相似度分值
    results, scores = retriever.retrieve(query_tokens, k=k)

    out = {}
    for i, cid in enumerate(cids):
        # 根据返回的索引 j，从证据 ID 列表（evidence_ids）中找回原始 ID
        out[cid] = [evidence_ids[j] for j in results[i]]

    return out

In [13]:
print("BM25s baseline(top-5)")
t0 = time.time()
TOP_K = 5

# 执行检索，这里的 TOP_K 沿用你之前设置的 5
predict_R2 = bm25s_retrieve(dev_claims, k=TOP_K)

print(f'  用时: {time.time() - t0:.1f}s\n')

# 评估 BM25 的检索效果
f_R2 = eval_retrieval(dev_claims, predict_R2)

BM25s baseline(top-5)


Split strings:   0%|          | 0/154 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/154 [00:00<?, ?it/s]

  用时: 0.6s

Mean Recall:    0.160281
Mean Precision: 0.089610
Mean F1-Score:  0.107689


# 优化bm25s的逻辑，使用sbert+加权

In [14]:

import torch
import time
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer, util

# 1. 定义设备（解决报错的关键点）
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. 加载 SBERT 预训练模型 (all-MiniLM-L6-v2) 并移动到 GPU
print("Loading SBERT rerank model...")
rerank_model = SentenceTransformer('all-MiniLM-L6-v2').to(device)
# --- 步骤 2: 定义混合检索重排序函数 ---
def hybrid_retrieve(claims_dict, k_coarse=50, k_fine=5, alpha=0.7):
    """
    k_coarse: 粗排召回数量 (建议 50)
    k_fine: 最终选出的证据数量 (比赛要求 5)
    alpha: 权重系数 (0.5 为平衡点)
    """
    cids = list(claims_dict.keys())
    texts = [claims_dict[c]['claim_text'] for c in cids]

    # 1. BM25 粗排 (Recall)
    query_tokens = bm25s.tokenize(texts, stopwords='en', stemmer=None)
    results, bm25_scores = retriever.retrieve(query_tokens, k=k_coarse)

    out = {}
    print(f"Reranking {len(cids)} claims...")

    # 循环处理每个 claim 进行精排
    for i, cid in enumerate(tqdm(cids)):
        candidate_indices = results[i]
        candidate_eids = [evidence_ids[idx] for idx in candidate_indices]
        candidate_texts = [evidence[eid] for eid in candidate_eids]

        # 2. SBERT 精排 (Reranking)
        claim_emb = rerank_model.encode(texts[i], convert_to_tensor=True)
        ev_embs = rerank_model.encode(candidate_texts, convert_to_tensor=True)

        # 计算 Claim 与候选证据的余弦相似度
        semantic_scores = util.cos_sim(claim_emb, ev_embs)[0]

        # 3. 加权融合 (Weighted Score)
        # 归一化 BM25 分数
        max_b25 = bm25_scores[i].max() if bm25_scores[i].max() > 0 else 1.0
        norm_bm25 = torch.tensor(bm25_scores[i] / max_b25).to(device)

        # 融合公式: alpha * BM25 + (1 - alpha) * SBERT
        final_scores = alpha * norm_bm25 + (1 - alpha) * semantic_scores

        # 选取最终得分最高的前 k_fine (5) 条
        top_k_idx = torch.topk(final_scores, k=k_fine).indices.cpu().numpy()
        out[cid] = [candidate_eids[idx] for idx in top_k_idx]

    return out

# --- 步骤 3: 在开发集上执行并评估优化后的效果 ---
print("\n[Hybrid Retrieval] Running Rerank on Dev Set...")
t_start = time.time()
# 得到优化后的检索结果
predict_R2_hybrid = hybrid_retrieve(dev_claims, k_coarse=50, k_fine=5, alpha=0.3)

print(f'Hybrid 检索用时: {time.time() - t_start:.1f}s\n')

# 调用你之前的评估函数，对比 F1-Score 是否有显著提升
f_R2_hybrid = eval_retrieval(dev_claims, predict_R2_hybrid)

Loading SBERT rerank model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[Hybrid Retrieval] Running Rerank on Dev Set...


Split strings:   0%|          | 0/154 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/154 [00:00<?, ?it/s]

Reranking 154 claims...


  0%|          | 0/154 [00:00<?, ?it/s]

Hybrid 检索用时: 5.0s

Mean Recall:    0.251623
Mean Precision: 0.142857
Mean F1-Score:  0.170068


In [15]:
# import gc, psutil

# # 删除不再需要的巨型变量，以释放内存空间
# del cos_word       # 删除 TF-IDF 的相似度矩阵
# del corpus_tokens  # 删除 BM25 分词后的语料库
# del retriever      # 删除 BM25 检索器对象

# # 显式调用垃圾回收器，立即清理被删除变量占用的内存
# gc.collect()

# classification

# baseline--BiLSTM

In [16]:
# classification
import torch
from torch.utils.data import Dataset, DataLoader

# 定义标签列表，对应项目中的四种分类结果
LABELS   = ['SUPPORTS', 'REFUTES', 'NOT_ENOUGH_INFO', 'DISPUTED']
# 构建标签与 ID 之间的双向映射字典，方便模型处理数字而非字符串
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

# 检查是否有可用的 GPU (CUDA)，如果有则使用 GPU，否则使用 CPU
device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

def make_input_text(claim_text, evidence_id_list, evidence_dict, max_evs=5):
    """
    把一条 claim + 它的若干条 evidence 拼成一个字符串，作为分类器的输入。
    用 [SEP] 显式分隔，这样 BiLSTM 能看到边界，Transformer 也能用。
    """
    # 取出前 max_evs 条证据的内容，并用句号连接
    ev_text = ' '.join([evidence_dict[eid] for eid in evidence_id_list[:max_evs]])
    # 将 claim 和拼接后的证据用分隔符 [SEP] 连在一起
    return claim_text + ' [SEP] ' + ev_text

Device: cuda


In [17]:
from collections import Counter

def simple_tokenize(text):
    """
    最朴素的分词：小写 + 按空格切分。
    Baseline 不引入 spacy/nltk 复杂分词，保持简单可复现。
    """
    return text.lower().split()

# 1. 统计训练集中所有词出现的频率
token_counter = Counter()
for cid, c in train_claims.items():
    # 使用上一步定义的 make_input_text 拼接 claim 和 evidence
    text = make_input_text(c['claim_text'], c['evidences'], evidence)
    # 更新词频统计
    token_counter.update(simple_tokenize(text))

# 定义词汇表的过滤参数
VOCAB_MIN_FREQ = 2      # 至少出现 2 次的词才进入词汇表
VOCAB_MAX_SIZE = 30000  # 词汇表最大容量为 3 万个词

# 2. 构建词汇表映射字典 (word -> id)
# <pad>: 用于填充长度不足的序列，通常 id 为 0
# <unk>: 代表词汇表之外的未知词（Unknown），通常 id 为 1
vocab = {'<pad>': 0, '<unk>': 1}
for word, cnt in token_counter.most_common(VOCAB_MAX_SIZE):
    if cnt < VOCAB_MIN_FREQ:
        break
    vocab[word] = len(vocab)

def encode(text, max_len=256):
    """
    文本 -> 定长 id 序列。短的右 pad，长的截断。
    """
    # 将词转换为 ID，如果不在词汇表中则使用 <unk> 的 ID (1)
    # 同时实现截断：只取前 max_len 个词
    ids = [vocab.get(w, 1) for w in simple_tokenize(text)][:max_len]

    # 实现填充 (Padding)：如果长度不足 max_len，在右侧补 0 (<pad>)
    ids = ids + [0] * (max_len - len(ids))
    return ids

In [18]:
class ClaimEvDatasetLSTM(Dataset):
    """
    把 (claim, evidence_list, label) 三元组封装成 Dataset。
    训练时 retrieval=None，自动用 Ground Truth (GT)；推理时传入 retrieval dict，用检索出的 evidence。
    """
    def __init__(self, claims_dict, evidence_dict, retrieval=None, max_len=256):
        self.cids      = list(claims_dict.keys())
        self.claims    = claims_dict
        self.evidence  = evidence_dict
        self.retrieval = retrieval
        self.max_len   = max_len
        # 自动判断数据集中是否包含标签（训练集有标签，测试集可能没有）
        self.has_label = 'claim_label' in next(iter(claims_dict.values()))

    def __len__(self):
        # 返回数据集的总条数
        return len(self.cids)

    def __getitem__(self, i):
        # 根据索引获取单条数据
        cid = self.cids[i]
        c   = self.claims[cid]

        # 逻辑判断：如果 retrieval 为空（训练阶段），使用答案中给出的真实证据 ID；
        # 如果 retrieval 不为空（测试阶段），使用你之前检索出来的证据 ID。
        ev_ids = c['evidences'] if self.retrieval is None else self.retrieval[cid]

        # 1. 拼接文本: Claim + [SEP] + Evidence
        text   = make_input_text(c['claim_text'], ev_ids, self.evidence)
        # 2. 文本编码: 将文字转为长度固定的 ID 序列，并转为 PyTorch 张量 (tensor)
        x      = torch.tensor(encode(text, self.max_len), dtype=torch.long)

        if self.has_label:
            # 3. 标签编码: 将文字标签转为数字 ID
            y = torch.tensor(LABEL2ID[c['claim_label']], dtype=torch.long)
            return x, y

        # 如果是预测阶段，返回输入 x 和 claim_id (cid) 方便最后写结果
        return x, cid

# 实例化数据集
train_ds_lstm = ClaimEvDatasetLSTM(train_claims, evidence, retrieval=None)
# 开发集使用之前检索出来的 predict_R1 结果作为输入
dev_ds_lstm   = ClaimEvDatasetLSTM(dev_claims,   evidence, retrieval=predict_R2)

# 使用 DataLoader 批量加载数据
train_loader_lstm = DataLoader(train_ds_lstm, batch_size=32, shuffle=True)
dev_loader_lstm   = DataLoader(dev_ds_lstm,   batch_size=32, shuffle=False)

In [19]:
# pytoch的模型搭建，固定写法
# dropout 层/ 正则化层 要不要加，以及顺序？ 改变量名
import torch.nn as nn

class BiLSTMClassifier(nn.Module):
    """
    三层结构：词嵌入 -> BiLSTM -> masked mean pool -> 线性 4 类输出。
    参数数量约 5-6M, T4 GPU 上 5 epoch 几分钟跑完。
    """
    def __init__(self, vocab_size, emb_dim=100, hidden=128, n_class=4, dropout=0.3):
        super().__init__()
        # 1. 词嵌入层：将单词 ID 映射为稠密向量，padding_idx=0 表示对 <pad> 进行零初始化
        self.emb      = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        # 2. 双向 LSTM 层：处理序列数据
        self.lstm     = nn.LSTM(emb_dim, hidden, batch_first=True, bidirectional=True)
        # 3. Dropout 层：防止过拟合
        self.dropout  = nn.Dropout(dropout)
        # 4. 全连接层：因为是双向的，隐藏层维度要乘以 2，最后输出 4 个类别的得分
        self.fc       = nn.Linear(hidden * 2, n_class)

    def forward(self, x):
        # 创建 mask 矩阵，排除掉填充位 <pad> (ID 为 0) 对平均值计算的影响
        mask   = (x != 0).unsqueeze(-1).float()
        # 词嵌入转换
        emb    = self.emb(x)
        # 通过 LSTM 得到每个时间步的输出 h
        h, _   = self.lstm(emb)

        # Masked Mean Pooling：对非填充部分的隐藏状态取平均值，得到整个句子的表示
        # 这种做法比只取最后一个时间步的效果通常更好
        pooled = (h * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

        # 通过全连接层得到分类对数几率 (logits)
        logits = self.fc(self.dropout(pooled))
        return logits

# 实例化模型并移动到 GPU (device)
model_C1 = BiLSTMClassifier(vocab_size=len(vocab)).to(device)
# 计算并打印模型参数量
n_params = sum(p.numel() for p in model_C1.parameters())
print(f'BiLSTM 参数量: {n_params:,}')

import torch.optim as optim
from collections import defaultdict

# 1. 计算类别权重 (Class Weights) 以处理数据不平衡问题
label_counts = Counter(c['claim_label'] for c in train_claims.values())
class_weights = torch.tensor(
    [len(train_claims) / (len(LABELS) * label_counts[l]) for l in LABELS],
    dtype=torch.float
).to(device)
print('Class weights:', dict(zip(LABELS, class_weights.cpu().tolist())))

# 2. 定义损失函数和优化器
# 使用加权交叉熵，让模型更加关注样本较少的类别
loss_fn   = nn.CrossEntropyLoss(weight=class_weights)
# 使用 Adam 优化器，学习率设为 1e-3
optimizer = optim.Adam(model_C1.parameters(), lr=1e-3)

EPOCHS = 6

# 3. 正式开始训练循环
for epoch in range(EPOCHS):
    model_C1.train() # 将模型设为训练模式
    epoch_losses = []

    for x, y in train_loader_lstm:
        # 将数据搬运到 GPU (device)
        x, y = x.to(device), y.to(device)

        # 前向传播：模型预测
        logits = model_C1(x)
        # 计算损失
        loss   = loss_fn(logits, y)

        # 反向传播与优化
        optimizer.zero_grad()  # 清空上一步的梯度
        loss.backward()        # 计算当前梯度
        # 梯度裁剪：防止梯度爆炸，这对 LSTM 这种循环神经网络非常重要
        torch.nn.utils.clip_grad_norm_(model_C1.parameters(), max_norm=5.0)
        optimizer.step()       # 更新模型参数

        epoch_losses.append(loss.item())

    # 打印每个 Epoch 的平均损失
    print(f'Epoch {epoch+1}/{EPOCHS}  train_loss = {np.mean(epoch_losses):.4f}')

BiLSTM 参数量: 967,748
Class weights: {'SUPPORTS': 0.5915221571922302, 'REFUTES': 1.5427135229110718, 'NOT_ENOUGH_INFO': 0.7953367829322815, 'DISPUTED': 2.475806474685669}
Epoch 1/6  train_loss = 1.3735
Epoch 2/6  train_loss = 1.2924
Epoch 3/6  train_loss = 1.1924
Epoch 4/6  train_loss = 1.0650
Epoch 5/6  train_loss = 0.9665
Epoch 6/6  train_loss = 0.8740


In [20]:
def predict_with_lstm(model, data_loader, dataset):
    """
    跑模型拿到每条 claim 的预测标签，返回 dict[claim_id] -> label_string。
    """
    model.eval()  # 将模型设为评估模式
    all_preds = []

    with torch.no_grad():  # 预测阶段不需要计算梯度，节省内存和计算资源
        for x, _ in data_loader:
            x = x.to(device)
            # 获取模型输出的对数几率 (logits)
            logits = model(x)
            # 取分值最大的类别索引作为预测结果
            preds  = logits.argmax(dim=-1).cpu().tolist()
            all_preds.extend(preds)

    # 将预测的数字 ID 映射回文字标签，并与 claim_id 对应
    return {dataset.cids[i]: ID2LABEL[p] for i, p in enumerate(all_preds)}

# 1. 运行预测函数获取开发集的预测结果
pred_label_C1 = predict_with_lstm(model_C1, dev_loader_lstm, dev_ds_lstm)

# 2. 计算预测正确的数量
correct = sum(1 for cid, plabel in pred_label_C1.items()
             if dev_claims[cid]['claim_label'] == plabel)

# 3. 计算并打印准确率 (Accuracy)
acc_C1 = correct / len(pred_label_C1)
print(f'C1 Classification Accuracy on dev: {acc_C1:.4f}')

# train的20，acc越低，可能是过拟合；epoch=3,acc也低；epoch=5，acc最高
#
# 最高是epoch=6, acc=0.3766
#

C1 Classification Accuracy on dev: 0.3896


# DistilBERT

In [21]:
# 1. 安装 Hugging Face 的 transformers 库
!pip install -q transformers

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import get_linear_schedule_with_warmup

# 2. 选择预训练模型：这里使用的是 DistilBERT（BERT 的轻量化版本）
# 它在保持高性能的同时，比 BERT 快 60%，体积小 40%
MODEL_NAME = 'distilbert-base-uncased'

# 3. 加载分词器和模型
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 加载专门用于序列分类的模型头
model_C2  = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,            # 对应你的 4 个标签
    id2label=ID2LABEL,      # 之前定义的 ID 到标签映射
    label2id=LABEL2ID,      # 之前定义的标签 到 ID 映射
).to(device)

# 4. 计算并打印参数量
n_params = sum(p.numel() for p in model_C2.parameters())
print(f'DistilBERT 参数量: {n_params:,}')


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBERT 参数量: 66,956,548


In [22]:
# baselien2比baseline1 参数量很多？考虑过拟合的问题？
class ClaimEvDatasetBERT(Dataset):
    """
    和 LSTM 版结构一样，但用 HuggingFace tokenizer 编码，不再自己拼 [SEP]。
    Tokenizer 会在 claim 和 evidence 中间自动插入 [SEP] token。
    """
    def __init__(self, claims_dict, evidence_dict, retrieval=None,
                 tokenizer=None, max_len=512):
        self.cids      = list(claims_dict.keys())
        self.claims    = claims_dict
        self.evidence  = evidence_dict
        self.retrieval = retrieval
        self.tokenizer = tokenizer
        self.max_len   = max_len
        self.has_label = 'claim_label' in next(iter(claims_dict.values()))

    def __len__(self):
        return len(self.cids)

    def __getitem__(self, i):
        cid = self.cids[i]
        c   = self.claims[cid]

        # 确定证据 ID 列表：训练用真实答案，开发/测试用检索结果
        ev_ids = c['evidences'] if self.retrieval is None else self.retrieval[cid]
        # 将前 5 条证据文本拼接成一个长字符串
        ev_text = ' '.join([self.evidence[e] for e in ev_ids[:5]])

        # 使用 BERT Tokenizer 进行编码
        enc = self.tokenizer(
            c['claim_text'],          # 第一个句子
            ev_text,                  # 第二个句子
            truncation=True,          # 超出长度自动截断
            padding='max_length',     # 长度不足自动填充
            max_length=self.max_len,  # 最大长度（256）
            return_tensors='pt',      # 返回 PyTorch 张量
        )

        # 移除多余的维度并转化为字典格式
        item = {k: v.squeeze(0) for k, v in enc.items()}

        if self.has_label:
            # 添加标签
            item['labels'] = torch.tensor(LABEL2ID[c['claim_label']], dtype=torch.long)

        return item

# 实例化数据集
train_ds_bert = ClaimEvDatasetBERT(train_claims, evidence, retrieval=None, tokenizer=tokenizer, max_len=512)
# 注意：这里开发集传入了 predict_R2，也就是你之前用 BM25 跑出来的检索结果
dev_ds_bert   = ClaimEvDatasetBERT(dev_claims,   evidence, retrieval=predict_R2, tokenizer=tokenizer, max_len=512)

# 实例化 DataLoader
# BERT 模型较大，batch_size 通常设小一点（比如 16）以防显存溢出
train_loader_bert = DataLoader(train_ds_bert, batch_size=16, shuffle=True)
dev_loader_bert   = DataLoader(dev_ds_bert,   batch_size=32, shuffle=False)

In [23]:
#名字/顺序换一下，基本都会这样写。'
# 运行不出来

In [24]:
# 1. 设置微调参数
EPOCHS_BERT = 5
LR_BERT     = 2e-5  # 预训练模型通常使用非常小的学习率

# 使用 AdamW 优化器，它比普通 Adam 更适合微调 Transformer 模型，包含了权重衰减 (Weight Decay)
optimizer_bert = torch.optim.AdamW(model_C2.parameters(), lr=LR_BERT, weight_decay=0.01)

# 计算总训练步数，用于设置学习率调度器
total_steps    = len(train_loader_bert) * EPOCHS_BERT

# 2. 设置学习率调度器 (Scheduler)
# 包含 Warmup 阶段：学习率在前 10% 的步数中从 0 增加到 2e-5，然后线性下降。这能防止模型在训练初期崩溃。
scheduler_bert = get_linear_schedule_with_warmup(
    optimizer_bert,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

# 沿用之前的类别权重损失函数
loss_fn_bert = nn.CrossEntropyLoss(weight=class_weights)

# 3. 正式开始训练
for epoch in range(EPOCHS_BERT):
    model_C2.train()
    epoch_losses = []

    for batch in train_loader_bert:
        # 将整个 batch 的数据（input_ids, attention_mask 等）搬运到 GPU
        batch  = {k: v.to(device) for k, v in batch.items()}
        labels = batch.pop('labels')  # 弹出标签用于计算 loss

        # 前向传播：**batch 会将字典解包为模型的参数输入
        outputs = model_C2(**batch)
        logits  = outputs.logits
        loss    = loss_fn_bert(logits, labels)

        # 反向传播
        optimizer_bert.zero_grad()
        loss.backward()

        # 梯度裁剪：Transformer 建议设为 1.0
        torch.nn.utils.clip_grad_norm_(model_C2.parameters(), max_norm=1.0)

        optimizer_bert.step()   # 更新参数
        scheduler_bert.step()   # 更新学习率

        epoch_losses.append(loss.item())

    print(f'Epoch {epoch+1}/{EPOCHS_BERT}  train_loss = {np.mean(epoch_losses):.4f}')

Epoch 1/5  train_loss = 1.3541
Epoch 2/5  train_loss = 1.1401
Epoch 3/5  train_loss = 0.9821
Epoch 4/5  train_loss = 0.8478
Epoch 5/5  train_loss = 0.7712


In [25]:
def predict_with_bert(model, data_loader, dataset):
    """
    跑 transformer 拿到每条 claim 的预测标签。
    """
    model.eval()  # 切换到评估模式，关闭 Dropout
    all_preds = []

    with torch.no_grad():  # 禁用梯度计算，节省显存
        for batch in data_loader:
            # 移除标签（如果存在），因为预测阶段不需要它
            batch.pop('labels', None)
            # 将数据移动到 GPU
            batch = {k: v.to(device) for k, v in batch.items()}

            # 前向传播：解包 batch 字典并获取 logits
            logits = model(**batch).logits
            # 获取得分最高的类别索引
            preds  = logits.argmax(dim=-1).cpu().tolist()
            all_preds.extend(preds)

    # 将结果映射回字典：{claim_id: label_string}
    return {dataset.cids[i]: ID2LABEL[p] for i, p in enumerate(all_preds)}

# 1. 运行预测
pred_label_C2 = predict_with_bert(model_C2, dev_loader_bert, dev_ds_bert)

# 2. 计算正确预测的数量
correct = sum(1 for cid, plabel in pred_label_C2.items()
             if dev_claims[cid]['claim_label'] == plabel)

# 3. 计算并打印准确率
acc_C2 = correct / len(pred_label_C2)
print(f'C2 Classification Accuracy on dev: {acc_C2:.4f}')

C2 Classification Accuracy on dev: 0.2143


# 输出test

# bm25+BiLSTM

In [26]:
# import json

# # 1. 使用 BM25 为测试集检索 Top-5 证据
# print("Retrieving evidence for test set using BM25...")
# # 这里的 test_claims 是你之前通过 json.load 加载的 test-claims-unlabelled.json
# predict_test_retrieval = bm25s_retrieve(test_claims, k=5)

# # 2. 准备测试集 Dataset 和 DataLoader
# # 注意：这里传入 retrieval=predict_test_retrieval 使用刚才检索到的证据
# test_ds_lstm = ClaimEvDatasetLSTM(
#     test_claims,
#     evidence,
#     retrieval=predict_test_retrieval,
#     max_len=256
# )
# test_loader_lstm = DataLoader(test_ds_lstm, batch_size=32, shuffle=False)

# # 3. 使用训练好的 BiLSTM 模型 (model_C1) 进行分类预测
# print("Predicting labels for test set using BiLSTM...")
# # 使用你定义的 predict_with_lstm 函数
# pred_label_test = predict_with_lstm(model_C1, test_loader_lstm, test_ds_lstm)

# # 4. 按照比赛要求的格式整合输出
# # 格式要求：{ "claim_id": { "claim_label": "...", "evidences": ["...", "..."] } }
# test_output = {}
# for cid in test_claims.keys():
#     test_output[cid] = {
#         "claim_label": pred_label_test[cid],
#         "evidences": predict_test_retrieval[cid]
#     }

# # 5. 保存为 json 文件
# output_path = 'test-output.json'
# with open(output_path, 'w', encoding='utf-8') as f:
#     json.dump(test_output, f, indent=4)

# print(f"Success! Test output saved to {output_path}")

# SBERT+BiLSTM

In [27]:
# # SBERT 精排+ BiLSTM
# import json

# # 1. 使用混合检索 (Hybrid Retrieval) 为测试集筛选证据
# # 相比基础 BM25，这一步引入了 SBERT 语义重排序，能显著提升检索质量
# print("Running Hybrid Retrieval for test set (BM25 + SBERT)...")
# # k_coarse=50 保证召回率，k_fine=5 保证精排准确度
# predict_test_hybrid = hybrid_retrieve(test_claims, k_coarse=50, k_fine=5, alpha=0.5)

# # 2. 准备测试集 Dataset 和 DataLoader
# # 使用混合检索得到的 predict_test_hybrid 作为输入
# test_ds_final = ClaimEvDatasetLSTM(
#     test_claims,
#     evidence,
#     retrieval=predict_test_hybrid,
#     max_len=256 # 保持与模型训练时的长度一致
# )
# test_loader_final = DataLoader(test_ds_final, batch_size=32, shuffle=False)

# # 3. 使用训练好的 BiLSTM 模型 (model_C1) 进行分类预测
# print("Predicting labels for test set using BiLSTM...")
# # 模型将基于更高质量的语义证据进行判断，有助于提升分类准确率
# pred_label_test_final = predict_with_lstm(model_C1, test_loader_final, test_ds_final)

# # 4. 按照比赛要求的格式整合输出
# test_output_final = {}
# for cid in test_claims.keys():
#     test_output_final[cid] = {
#         "claim_label": pred_label_test_final[cid],
#         "evidences": predict_test_hybrid[cid]
#     }

# # 5. 保存为 json 文件
# output_path = 'test-output.json'
# with open(output_path, 'w', encoding='utf-8') as f:
#     json.dump(test_output_final, f, indent=4)

# print(f"\nSuccess! Hybrid output saved to {output_path}")

# SBERT+DistilBERT



In [28]:
import json
import torch
from torch.utils.data import DataLoader

# --- 步骤 1: 使用混合检索 (Hybrid Retrieval) 筛选证据 ---
# 建议使用你在 Dev 集上调试出的最佳 alpha 值（例如 0.2 或 0.3）
BEST_ALPHA = 0.3
print(f"Running Hybrid Retrieval for test set (BM25 + SBERT, alpha={BEST_ALPHA})...")

# 使用你之前定义的 hybrid_retrieve 函数
predict_test_hybrid = hybrid_retrieve(test_claims, k_coarse=50, k_fine=5, alpha=BEST_ALPHA)

# --- 步骤 2: 准备基于 BERT 的测试集和加载器 ---
# 注意：这里必须使用 ClaimEvDatasetBERT 类，因为它包含了 BERT 的 Tokenizer 逻辑
test_ds_bert_final = ClaimEvDatasetBERT(
    test_claims,
    evidence,
    retrieval=predict_test_hybrid,
    tokenizer=tokenizer,  # 使用之前加载的 distilbert-base-uncased tokenizer
    max_len=512           # DistilBERT 建议使用 512 以包含更多上下文信息
)

# BERT 模型较大，batch_size 建议设为 32 或更小
test_loader_bert_final = DataLoader(test_ds_bert_final, batch_size=32, shuffle=False)

# --- 步骤 3: 使用微调后的 DistilBERT (model_C2) 进行预测 ---
print("Predicting labels for test set using DistilBERT...")
# 使用你定义的 predict_with_bert 函数
pred_label_test_bert = predict_with_bert(model_C2, test_loader_bert_final, test_ds_bert_final)

# --- 步骤 4: 整合并保存输出结果 ---
test_output_bert = {}
for cid in test_claims.keys():
    test_output_bert[cid] = {
        "claim_label": pred_label_test_bert[cid],
        "evidences": predict_test_hybrid[cid]
    }

# 保存文件
output_path = 'test-output.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(test_output_bert, f, indent=4)

print(f"\nSuccess! DistilBERT + Hybrid output saved to {output_path}")

Running Hybrid Retrieval for test set (BM25 + SBERT, alpha=0.3)...


Split strings:   0%|          | 0/153 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/153 [00:00<?, ?it/s]

Reranking 153 claims...


  0%|          | 0/153 [00:00<?, ?it/s]

Predicting labels for test set using DistilBERT...

Success! DistilBERT + Hybrid output saved to test-output.json


# 2.Model Implementation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

# 3.Testing and Evaluation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

## Object Oriented Programming codes here

*You can use multiple code snippets. Just add more if needed*